# Notebook 2: Macro Stress Testing & ECL Estimation
## Indian MSME Credit Risk Framework

**Objective:** Simulate how portfolio credit losses escalate under three
macro scenarios — Base, Moderate Stress, and Severe Stress — using an
IFRS 9-aligned PD x LGD x EAD framework.

**Synthetic MSME Features Added:**
- Sector (Manufacturing, Retail & Trade, Agri & Allied, Services, Construction)
- Region (North, South, East, West, Central)
- GST Compliance Flag
- Macro Indicators (GDP Growth, Repo Rate, Unemployment)

**Scenario Assumptions:**
| Scenario | GDP Growth | Repo Rate | Unemployment |
|----------|-----------|-----------|--------------|
| Base | 6.5% | 6.5% | 7.8% |
| Moderate Stress | 4.0% | 7.5% | 9.5% |
| Severe Stress | 1.5% | 9.0% | 12.0% |

**Key Results:**
- Base ECL Rate: 8.00%
- Moderate Stress ECL Rate: 11.21%
- Severe Stress ECL Rate: 16.01%
- High Risk segment PD doubles under severe stress (47% → 94%)

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# Load scored portfolio
df = pd.read_csv('/content/drive/MyDrive/Credit_Risk_Framework/Outputs/scored_portfolio.csv')

print(df.shape)
print(df.columns.tolist())
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(6000, 19)
['LIMIT_BAL', 'AGE', 'SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'UTIL_RATIO', 'PAY_RATIO', 'AVG_BILL', 'AVG_PAY', 'MAX_DELAY', 'SCORE', 'DEFAULT', 'RISK_BAND']
   LIMIT_BAL  AGE  SEX  EDUCATION  MARRIAGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0      50000   46    1          2         2     -1     -1     -1     -1   
1     150000   31    1          1         1     -1     -1     -2     -2   
2      50000   25    1          2         2      0      0      0      0   
3     290000   25    2          1         2      0      0      0      0   
4     500000   27    2          2         1     -2     -2     -2     -2   

   PAY_5  PAY_6  UTIL_RATIO  PAY_RATIO       AVG_BILL       AVG_PAY  \
0      0      0    0.030799  16.911097    6055.666667   6076.166667   
1     -2     -1    0.099999   0.000000    4449.000000   6949.000

In [8]:
import random
random.seed(42)
np.random.seed(42)

# 1. Sector — Indian MSME sectors
sectors = ['Manufacturing', 'Retail & Trade', 'Agri & Allied', 'Services', 'Construction']
sector_weights = [0.25, 0.30, 0.15, 0.20, 0.10]
df['SECTOR'] = np.random.choice(sectors, size=len(df), p=sector_weights)

# 2. Geography
regions = ['North', 'South', 'East', 'West', 'Central']
df['REGION'] = np.random.choice(regions, size=len(df))

# 3. GST Compliance Flag — higher risk borrowers less likely to be compliant
df['GST_COMPLIANT'] = np.where(
    df['RISK_BAND'] == 'High Risk',
    np.random.choice([0, 1], size=len(df), p=[0.6, 0.4]),
    np.where(df['RISK_BAND'] == 'Medium Risk',
             np.random.choice([0, 1], size=len(df), p=[0.3, 0.7]),
             np.random.choice([0, 1], size=len(df), p=[0.1, 0.9]))
)

# 4. Macro Indicators — same for all rows, represents current macro environment
df['GDP_GROWTH'] = 6.5      # RBI baseline GDP growth %
df['REPO_RATE'] = 6.5       # RBI repo rate %
df['UNEMPLOYMENT'] = 7.8    # Baseline unemployment %

print(df[['SECTOR', 'REGION', 'GST_COMPLIANT', 'GDP_GROWTH', 'REPO_RATE', 'UNEMPLOYMENT']].head(10))
print("\nSector distribution:\n", df['SECTOR'].value_counts())
print("\nGST Compliance by Risk Band:\n", df.groupby('RISK_BAND')['GST_COMPLIANT'].mean())

           SECTOR   REGION  GST_COMPLIANT  GDP_GROWTH  REPO_RATE  UNEMPLOYMENT
0  Retail & Trade    North              1         6.5        6.5           7.8
1    Construction    North              1         6.5        6.5           7.8
2        Services     East              1         6.5        6.5           7.8
3   Agri & Allied  Central              1         6.5        6.5           7.8
4   Manufacturing  Central              1         6.5        6.5           7.8
5   Manufacturing     East              1         6.5        6.5           7.8
6   Manufacturing     West              1         6.5        6.5           7.8
7        Services    North              1         6.5        6.5           7.8
8   Agri & Allied    North              1         6.5        6.5           7.8
9        Services     East              1         6.5        6.5           7.8

Sector distribution:
 SECTOR
Retail & Trade    1768
Manufacturing     1534
Services          1175
Agri & Allied      936
Construct

In [9]:
# Define three macro scenarios
scenarios = {
    'Base': {
        'GDP_GROWTH': 6.5,
        'REPO_RATE': 6.5,
        'UNEMPLOYMENT': 7.8,
        'PD_MULTIPLIER': 1.0
    },
    'Moderate Stress': {
        'GDP_GROWTH': 4.0,
        'REPO_RATE': 7.5,
        'UNEMPLOYMENT': 9.5,
        'PD_MULTIPLIER': 1.4
    },
    'Severe Stress': {
        'GDP_GROWTH': 1.5,
        'REPO_RATE': 9.0,
        'UNEMPLOYMENT': 12.0,
        'PD_MULTIPLIER': 2.0
    }
}

# Base PD by risk band
base_pd = {
    'Low Risk': 0.078847,
    'Medium Risk': 0.164896,
    'High Risk': 0.470988
}

# LGD and EAD assumptions
LGD = 0.45  # standard retail lending assumption
df['EAD'] = df['LIMIT_BAL']  # exposure at default = credit limit

print("Scenarios defined successfully.")
print("\nBase PD by Risk Band:")
for band, pd_val in base_pd.items():
    print(f"  {band}: {pd_val:.2%}")

Scenarios defined successfully.

Base PD by Risk Band:
  Low Risk: 7.88%
  Medium Risk: 16.49%
  High Risk: 47.10%


In [10]:
# Run stress test
results = []

for scenario_name, params in scenarios.items():
    for band in ['Low Risk', 'Medium Risk', 'High Risk']:
        segment = df[df['RISK_BAND'] == band].copy()

        # Stressed PD
        stressed_pd = min(base_pd[band] * params['PD_MULTIPLIER'], 1.0)

        # Expected Credit Loss = PD x LGD x EAD
        segment['PD'] = stressed_pd
        segment['LGD'] = LGD
        segment['ECL'] = segment['PD'] * segment['LGD'] * segment['EAD']

        total_ead = segment['EAD'].sum()
        total_ecl = segment['ECL'].sum()
        ecl_rate = total_ecl / total_ead

        results.append({
            'Scenario': scenario_name,
            'Risk Band': band,
            'Borrowers': len(segment),
            'Stressed PD': round(stressed_pd * 100, 2),
            'Total EAD (TWD)': round(total_ead, 0),
            'Total ECL (TWD)': round(total_ecl, 0),
            'ECL Rate (%)': round(ecl_rate * 100, 2)
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

       Scenario   Risk Band  Borrowers  Stressed PD  Total EAD (TWD)  Total ECL (TWD)  ECL Rate (%)
           Base    Low Risk       1839         7.88        489137680       17355167.0          3.55
           Base Medium Risk       2541        16.49        338420000       25111847.0          7.42
           Base   High Risk       1620        47.10        180220000       38196656.0         21.19
Moderate Stress    Low Risk       1839        11.04        489137680       24297234.0          4.97
Moderate Stress Medium Risk       2541        23.09        338420000       35156586.0         10.39
Moderate Stress   High Risk       1620        65.94        180220000       53475318.0         29.67
  Severe Stress    Low Risk       1839        15.77        489137680       34710335.0          7.10
  Severe Stress Medium Risk       2541        32.98        338420000       50223694.0         14.84
  Severe Stress   High Risk       1620        94.20        180220000       76393312.0         42.39


In [12]:
# Portfolio level summary
portfolio_summary = results_df.groupby('Scenario').agg(
    Total_EAD=('Total EAD (TWD)', 'sum'),
    Total_ECL=('Total ECL (TWD)', 'sum')
).reset_index()

portfolio_summary['Portfolio ECL Rate (%)'] = round(
    portfolio_summary['Total_ECL'] / portfolio_summary['Total_EAD'] * 100, 2)

print("Portfolio Level Summary:")
print(portfolio_summary.to_string(index=False))

# Save results
results_df.to_csv('/content/drive/MyDrive/Credit_Risk_Framework/Outputs/stress_results.csv', index=False)
df.to_csv('/content/drive/MyDrive/Credit_Risk_Framework/Outputs/enriched_portfolio.csv', index=False)
print("\nFiles saved.")

Portfolio Level Summary:
       Scenario  Total_EAD   Total_ECL  Portfolio ECL Rate (%)
           Base 1007777680  80663670.0                    8.00
Moderate Stress 1007777680 112929138.0                   11.21
  Severe Stress 1007777680 161327341.0                   16.01

Files saved.
